In [1]:
# Importing Libraries
import quandl
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.api as sm # Using .api imports the public access version of statsmodels, which is a library that handles 
# statistical models.
import os
import warnings # This is a library that handles warnings.

warnings.filterwarnings("ignore") # Disable deprecation warnings that could indicate, for instance, a suspended library or 
# feature. These are more relevant to developers and very seldom to analysts.

plt.style.use('fivethirtyeight') # This is a styling option for how your plots will appear. More examples here:
# https://matplotlib.org/3.2.1/tutorials/introductory/customizing.html
# https://matplotlib.org/3.1.0/gallery/style_sheets/fivethirtyeight.html

In [8]:
##Step 2. Importing Data
df = pd.read_pickle('/Users/javieraquezada/Desktop/Retail Analysis/02 Data/cleaned_sample.pkl')

In [9]:
df.head(5)

,Transaction_ID,Customer_ID,Name,Email,Phone,Address,City,State,Zipcode,Country,...,Total_Amount,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings,products
77037,8154457,18745,Tara Wright,Jeffrey100@gmail.com,6.443965e+09,474 Kevin Road Apt. 786,Chicago,Connecticut,14063.0,USA,...,2350.802482,Books,HarperCollins,Fiction,Excellent,Standard,PayPal,Processing,5.0,Historical fiction
106765,8896002,82300,Daniel Patton,Cheyenne19@gmail.com,1.792779e+09,717 Timothy Prairie,Brisbane,New South Wales,82508.0,Australia,...,605.991344,Clothing,Zara,Jeans,Excellent,Same-Day,Debit Card,Shipped,5.0,Wide-leg jeans
187652,1820950,49584,Kathleen Watson,Billy97@gmail.com,2.066960e+09,2069 Nicholas Prairie Suite 877,St. John's,Ontario,61208.0,Canada,...,676.659578,Grocery,Pepsi,Water,Bad,Standard,Credit Card,Shipped,1.0,Mineral water
184582,9448636,74580,Steven Sullivan,Gabriel71@gmail.com,8.490832e+09,912 Randall Manor,Saskatoon,Ontario,53361.0,Canada,...,272.016496,Home Decor,Bed Bath & Beyond,Bedding,Bad,Standard,Credit Card,Shipped,1.0,Mattress topper
290216,8088692,72061,Heather Yu,Daniel32@gmail.com,7.141601e+09,1297 Willis Drive,Halifax,Ontario,64745.0,Canada,...,2394.895305,Grocery,Nestle,Snacks,Good,Standard,Debit Card,Shipped,4.0,Chips


In [10]:
df.shape

(9759, 30)

In [11]:
df.columns

Index(['Transaction_ID', 'Customer_ID', 'Name', 'Email', 'Phone', 'Address',
       'City', 'State', 'Zipcode', 'Country', 'Age', 'Gender', 'Income',
       'Customer_Segment', 'Date', 'Year', 'Month', 'Time', 'Total_Purchases',
       'Amount', 'Total_Amount', 'Product_Category', 'Product_Brand',
       'Product_Type', 'Feedback', 'Shipping_Method', 'Payment_Method',
       'Order_Status', 'Ratings', 'products'],
      dtype='object')

In [6]:
type(data)

pandas.core.frame.DataFrame

In [41]:
# Converting date to a datetime fromat
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
## 2 Sorting by date just in case
df = df.sort_values('Date')

In [42]:
# Step 1: Selecting relevant columns
columns_needed = [
    'Customer_ID', 'Gender', 'Age', 'Income', 'Customer_Segment',
    'Country', 'City',
    'Product_Category', 'Product_Brand', 'Product_Type',
    'Total_Purchases', 'Total_Amount', 'Ratings', 'Feedback',
    'Date', 'Month', 'Year',
    'Shipping_Method', 'Payment_Method', 'Order_Status','Feedback'
]

In [43]:
# Step 2: Filtering dataset to keep only those columns
clean_df = df[columns_needed].dropna()

In [44]:
clean_df.head

<bound method NDFrame.head of         Customer_ID  Gender   Age  Income Customer_Segment  Country  \
250190        43921    Male  64.0     Low          Regular   Canada   
103341        84812    Male  46.0    High              New       UK   
264280        92062    Male  33.0     Low          Premium  Germany   
186559        83193    Male  69.0  Medium          Regular   Canada   
228739        50940    Male  48.0  Medium          Regular      USA   
...             ...     ...   ...     ...              ...      ...   
115226        98687  Female  46.0  Medium          Regular      USA   
97935         36936  Female  46.0    High          Regular      USA   
283414        58423  Female  21.0    High          Premium      USA   
126815        19264  Female  48.0  Medium          Premium   Canada   
277650        54563    Male  28.0    High          Premium  Germany   

              City Product_Category Product_Brand  \
250190      Oshawa       Home Decor    Home Depot   
103341     

In [45]:
clean_df.columns

Index(['Customer_ID', 'Gender', 'Age', 'Income', 'Customer_Segment', 'Country',
       'City', 'Product_Category', 'Product_Brand', 'Product_Type',
       'Total_Purchases', 'Total_Amount', 'Ratings', 'Feedback', 'Date',
       'Month', 'Year', 'Shipping_Method', 'Payment_Method', 'Order_Status',
       'Feedback'],
      dtype='object')

In [46]:
# Check for missing values in each column
missing_values = df[columns_needed].isnull().sum().sort_values(ascending=False)

# Show only columns with missing values
missing_values = missing_values[missing_values > 0]

# Display results
print("🔎 Columns with Missing Values:\n")
print(missing_values)

🔎 Columns with Missing Values:

Series([], dtype: int64)


In [48]:
clean_df.dtypes

Customer_ID                  Int64
Gender                      object
Age                        float64
Income                      object
Customer_Segment            object
Country                     object
City                        object
Product_Category            object
Product_Brand               object
Product_Type                object
Total_Purchases            float64
Total_Amount               float64
Ratings                    float64
Feedback                    object
Date                datetime64[ns]
Month                     category
Year                       float64
Shipping_Method             object
Payment_Method              object
Order_Status                object
Feedback                    object
dtype: object

In [50]:
clean_df.head(20)


,Customer_ID,Gender,Age,Income,Customer_Segment,Country,City,Product_Category,Product_Brand,Product_Type,...,Total_Amount,Ratings,Feedback,Date,Month,Year,Shipping_Method,Payment_Method,Order_Status,Feedback
250190,43921,Male,64.0,Low,Regular,Canada,Oshawa,Home Decor,Home Depot,Decorations,...,2418.970997,4.0,Good,2023-03-01,March,2023.0,Same-Day,Credit Card,Shipped,Good
103341,84812,Male,46.0,High,New,UK,Glasgow,Home Decor,IKEA,Furniture,...,2903.437076,4.0,Excellent,2023-03-01,March,2023.0,Express,Debit Card,Shipped,Excellent
264280,92062,Male,33.0,Low,Premium,Germany,Wuppertal,Electronics,Mitsubhisi,Mitsubishi 1.5 Ton 3 Star Split AC,...,916.453503,4.0,Good,2023-03-01,April,2023.0,Same-Day,Credit Card,Delivered,Good
186559,83193,Male,69.0,Medium,Regular,Canada,London,Books,Random House,Fiction,...,351.990558,1.0,Bad,2023-03-01,March,2023.0,Standard,PayPal,Delivered,Bad
228739,50940,Male,48.0,Medium,Regular,USA,San Diego,Home Decor,Home Depot,Furniture,...,67.780750,4.0,Excellent,2023-03-01,March,2023.0,Standard,Cash,Pending,Excellent
189106,89702,Male,53.0,Low,New,Germany,Düsseldorf,Books,Random House,Non-Fiction,...,483.314915,4.0,Good,2023-03-01,March,2023.0,Express,Cash,Processing,Good
136056,95025,Male,26.0,High,Regular,Germany,Duisburg,Electronics,Whirepool,Fridge,...,1354.062845,4.0,Good,2023-03-01,January,2023.0,Express,Debit Card,Delivered,Good
59253,28253,Female,20.0,Medium,New,Canada,St. John's,Electronics,Samsung,Smartphone,...,691.198127,5.0,Excellent,2023-03-01,March,2023.0,Express,PayPal,Processing,Excellent
10030,88868,Male,19.0,Medium,Regular,UK,Portsmouth,Home Decor,Home Depot,Decorations,...,3095.936582,2.0,Average,2023-03-01,March,2023.0,Standard,Cash,Delivered,Average
84878,14478,Female,34.0,Low,Regular,USA,Chicago,Grocery,Pepsi,Water,...,3730.281903,3.0,Good,2023-03-01,March,2023.0,Standard,Credit Card,Processing,Good


In [51]:
## Fixing Data types

In [55]:
#Changing Year column data type from float 64 to int
clean_df['Year'] = clean_df['Year'].astype('int64')

In [56]:
# Fixing duplicated column 
clean_df = clean_df.loc[:, ~clean_df.columns.duplicated()]

In [57]:
#Converting Month as string for visualizations.
clean_df['Month'] = clean_df['Date'].dt.month_name()

In [58]:
from pandas.api.types import CategoricalDtype

month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

clean_df['Month'] = pd.Categorical(clean_df['Date'].dt.month_name(), categories=month_order, ordered=True)

In [59]:
clean_df.dtypes

Customer_ID                  Int64
Gender                      object
Age                        float64
Income                      object
Customer_Segment            object
Country                     object
City                        object
Product_Category            object
Product_Brand               object
Product_Type                object
Total_Purchases            float64
Total_Amount               float64
Ratings                    float64
Feedback                    object
Date                datetime64[ns]
Month                     category
Year                         int64
Shipping_Method             object
Payment_Method              object
Order_Status                object
dtype: object

In [60]:
#Mapping income levels to numeric values for viz
clean_df['Income'].unique()

array(['Low', 'High', 'Medium'], dtype=object)

In [61]:
# Mapping income levels to numeric values
income_map = {
    'Low': 30000,
    'Medium': 60000,
    'High': 90000
}

# Apply mapping
clean_df['Income'] = clean_df['Income'].replace(income_map).astype(float)

In [62]:
clean_df['Income'].dtype

dtype('float64')

In [63]:
clean_df.dtypes

Customer_ID                  Int64
Gender                      object
Age                        float64
Income                     float64
Customer_Segment            object
Country                     object
City                        object
Product_Category            object
Product_Brand               object
Product_Type                object
Total_Purchases            float64
Total_Amount               float64
Ratings                    float64
Feedback                    object
Date                datetime64[ns]
Month                     category
Year                         int64
Shipping_Method             object
Payment_Method              object
Order_Status                object
dtype: object

In [64]:
clean_df.head(20)

,Customer_ID,Gender,Age,Income,Customer_Segment,Country,City,Product_Category,Product_Brand,Product_Type,Total_Purchases,Total_Amount,Ratings,Feedback,Date,Month,Year,Shipping_Method,Payment_Method,Order_Status
250190,43921,Male,64.0,30000.0,Regular,Canada,Oshawa,Home Decor,Home Depot,Decorations,8.0,2418.970997,4.0,Good,2023-03-01,March,2023,Same-Day,Credit Card,Shipped
103341,84812,Male,46.0,90000.0,New,UK,Glasgow,Home Decor,IKEA,Furniture,8.0,2903.437076,4.0,Excellent,2023-03-01,March,2023,Express,Debit Card,Shipped
264280,92062,Male,33.0,30000.0,Premium,Germany,Wuppertal,Electronics,Mitsubhisi,Mitsubishi 1.5 Ton 3 Star Split AC,6.0,916.453503,4.0,Good,2023-03-01,March,2023,Same-Day,Credit Card,Delivered
186559,83193,Male,69.0,60000.0,Regular,Canada,London,Books,Random House,Fiction,1.0,351.990558,1.0,Bad,2023-03-01,March,2023,Standard,PayPal,Delivered
228739,50940,Male,48.0,60000.0,Regular,USA,San Diego,Home Decor,Home Depot,Furniture,3.0,67.780750,4.0,Excellent,2023-03-01,March,2023,Standard,Cash,Pending
189106,89702,Male,53.0,30000.0,New,Germany,Düsseldorf,Books,Random House,Non-Fiction,4.0,483.314915,4.0,Good,2023-03-01,March,2023,Express,Cash,Processing
136056,95025,Male,26.0,90000.0,Regular,Germany,Duisburg,Electronics,Whirepool,Fridge,3.0,1354.062845,4.0,Good,2023-03-01,March,2023,Express,Debit Card,Delivered
59253,28253,Female,20.0,60000.0,New,Canada,St. John's,Electronics,Samsung,Smartphone,2.0,691.198127,5.0,Excellent,2023-03-01,March,2023,Express,PayPal,Processing
10030,88868,Male,19.0,60000.0,Regular,UK,Portsmouth,Home Decor,Home Depot,Decorations,8.0,3095.936582,2.0,Average,2023-03-01,March,2023,Standard,Cash,Delivered
84878,14478,Female,34.0,30000.0,Regular,USA,Chicago,Grocery,Pepsi,Water,9.0,3730.281903,3.0,Good,2023-03-01,March,2023,Standard,Credit Card,Processing


In [65]:
# Creating  month name column for viz
clean_df['Quarter'] = clean_df['Date'].dt.to_period('Q')

In [66]:
clean_df.head(20)

,Customer_ID,Gender,Age,Income,Customer_Segment,Country,City,Product_Category,Product_Brand,Product_Type,...,Total_Amount,Ratings,Feedback,Date,Month,Year,Shipping_Method,Payment_Method,Order_Status,Quarter
250190,43921,Male,64.0,30000.0,Regular,Canada,Oshawa,Home Decor,Home Depot,Decorations,...,2418.970997,4.0,Good,2023-03-01,March,2023,Same-Day,Credit Card,Shipped,2023Q1
103341,84812,Male,46.0,90000.0,New,UK,Glasgow,Home Decor,IKEA,Furniture,...,2903.437076,4.0,Excellent,2023-03-01,March,2023,Express,Debit Card,Shipped,2023Q1
264280,92062,Male,33.0,30000.0,Premium,Germany,Wuppertal,Electronics,Mitsubhisi,Mitsubishi 1.5 Ton 3 Star Split AC,...,916.453503,4.0,Good,2023-03-01,March,2023,Same-Day,Credit Card,Delivered,2023Q1
186559,83193,Male,69.0,60000.0,Regular,Canada,London,Books,Random House,Fiction,...,351.990558,1.0,Bad,2023-03-01,March,2023,Standard,PayPal,Delivered,2023Q1
228739,50940,Male,48.0,60000.0,Regular,USA,San Diego,Home Decor,Home Depot,Furniture,...,67.780750,4.0,Excellent,2023-03-01,March,2023,Standard,Cash,Pending,2023Q1
189106,89702,Male,53.0,30000.0,New,Germany,Düsseldorf,Books,Random House,Non-Fiction,...,483.314915,4.0,Good,2023-03-01,March,2023,Express,Cash,Processing,2023Q1
136056,95025,Male,26.0,90000.0,Regular,Germany,Duisburg,Electronics,Whirepool,Fridge,...,1354.062845,4.0,Good,2023-03-01,March,2023,Express,Debit Card,Delivered,2023Q1
59253,28253,Female,20.0,60000.0,New,Canada,St. John's,Electronics,Samsung,Smartphone,...,691.198127,5.0,Excellent,2023-03-01,March,2023,Express,PayPal,Processing,2023Q1
10030,88868,Male,19.0,60000.0,Regular,UK,Portsmouth,Home Decor,Home Depot,Decorations,...,3095.936582,2.0,Average,2023-03-01,March,2023,Standard,Cash,Delivered,2023Q1
84878,14478,Female,34.0,30000.0,Regular,USA,Chicago,Grocery,Pepsi,Water,...,3730.281903,3.0,Good,2023-03-01,March,2023,Standard,Credit Card,Processing,2023Q1


In [67]:
print(clean_df.dtypes)

Customer_ID                  Int64
Gender                      object
Age                        float64
Income                     float64
Customer_Segment            object
Country                     object
City                        object
Product_Category            object
Product_Brand               object
Product_Type                object
Total_Purchases            float64
Total_Amount               float64
Ratings                    float64
Feedback                    object
Date                datetime64[ns]
Month                     category
Year                         int64
Shipping_Method             object
Payment_Method              object
Order_Status                object
Quarter              period[Q-DEC]
dtype: object


In [ ]:
# Data types ready for viz.

In [68]:
print(clean_df.isnull().sum())

Customer_ID         0
Gender              0
Age                 0
Income              0
Customer_Segment    0
Country             0
City                0
Product_Category    0
Product_Brand       0
Product_Type        0
Total_Purchases     0
Total_Amount        0
Ratings             0
Feedback            0
Date                0
Month               0
Year                0
Shipping_Method     0
Payment_Method      0
Order_Status        0
Quarter             0
dtype: int64


In [ ]:
# No null Values

In [69]:
print(clean_df.duplicated().sum())

0


In [ ]:
# NO duplicates

In [70]:
clean_df['Gender'] = clean_df['Gender'].str.strip().str.title()
clean_df['Customer_Segment'] = clean_df['Customer_Segment'].str.strip().str.title()

In [ ]:
# Spaces and details fixed.

In [71]:
print(clean_df[~clean_df['Ratings'].between(1, 5)])

Empty DataFrame
Columns: [Customer_ID, Gender, Age, Income, Customer_Segment, Country, City, Product_Category, Product_Brand, Product_Type, Total_Purchases, Total_Amount, Ratings, Feedback, Date, Month, Year, Shipping_Method, Payment_Method, Order_Status, Quarter]
Index: []

[0 rows x 21 columns]


In [72]:
# All values are within 1 to 5.

In [73]:
#Checking Date Ranges.
print(clean_df['Date'].min(), clean_df['Date'].max())

2023-03-01 00:00:00 2024-02-29 00:00:00


In [74]:
print(clean_df['Date'].min())
print(clean_df['Date'].max())

2023-03-01 00:00:00
2024-02-29 00:00:00


In [75]:
print(clean_df.head())

        Customer_ID Gender   Age   Income Customer_Segment  Country  \
250190        43921   Male  64.0  30000.0          Regular   Canada   
103341        84812   Male  46.0  90000.0              New       UK   
264280        92062   Male  33.0  30000.0          Premium  Germany   
186559        83193   Male  69.0  60000.0          Regular   Canada   
228739        50940   Male  48.0  60000.0          Regular      USA   

             City Product_Category Product_Brand  \
250190     Oshawa       Home Decor    Home Depot   
103341    Glasgow       Home Decor          IKEA   
264280  Wuppertal      Electronics    Mitsubhisi   
186559     London            Books  Random House   
228739  San Diego       Home Decor    Home Depot   

                              Product_Type  ...  Total_Amount  Ratings  \
250190                         Decorations  ...   2418.970997      4.0   
103341                           Furniture  ...   2903.437076      4.0   
264280  Mitsubishi 1.5 Ton 3 Star Spli

In [76]:
# Final checks
#Dates converted to datetime, with Month, Year, and Quarter columns for time grouping
#Numeric fields like Age, Income, Total_Amount, Ratings are properly typed
#Categorical fields like Gender, Customer_Segment, Country, City, Product_Category, Payment_Method, Order_Status are intact
#Feedback is there for qualitative insights


In [77]:
# Exporting dataframe as pickle just in case
clean_df.to_pickle('/Users/javieraquezada/Desktop/Retail Analysis/02 Data/cleaned_for_viz.pkl')

In [78]:
# Exporting dataframe as CSV for Tableau.
clean_df.to_csv('/Users/javieraquezada/Desktop/Retail Analysis/02 Data/cleaned_for_tableau.csv', index=False)